# Agency RMBS Competing Hazards SMM Example

This notebook creates a synthetic monthly panel (2,000 loans over 5 years) with two competing prepayment hazards:
- **Refinance**: driven by rate incentive and FICO
- **Buyout**: driven by LTV and FICO

Each hazard is modeled with a logistic model using smoothing splines, then combined into a global monthly SMM estimate.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import statsmodels.formula.api as smf
import statsmodels.api as sm

np.random.seed(7)


In [ ]:
N_LOANS = 2000
N_MONTHS = 60

loan_ids = np.arange(N_LOANS)
fico = np.clip(np.random.normal(loc=714, scale=100, size=N_LOANS), 300, 850)
ltv = np.clip(np.random.normal(loc=0.75, scale=0.25, size=N_LOANS), 0.01, 1.0)
base_incentive = np.random.normal(loc=0.0, scale=0.6, size=N_LOANS)

alive = np.ones(N_LOANS, dtype=bool)
rows = []

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

for month in range(1, N_MONTHS + 1):
    active_idx = np.where(alive)[0]
    if len(active_idx) == 0:
        break

    seasonal = 0.15 * np.sin(2 * np.pi * month / 12)
    incentive = base_incentive[active_idx] + seasonal + np.random.normal(0, 0.25, size=len(active_idx))

    refi_score = -2.9 + 2.1 * incentive + 0.006 * (fico[active_idx] - 714) / 10
    buyout_score = -3.8 - 2.8 * (ltv[active_idx] - 0.75) - 0.007 * (fico[active_idx] - 714) / 10

    lambda_refi = np.exp(refi_score)
    lambda_buyout = np.exp(buyout_score)
    lambda_total = lambda_refi + lambda_buyout

    smm_total = 1 - np.exp(-lambda_total)
    p_refi = smm_total * (lambda_refi / lambda_total)
    p_buyout = smm_total * (lambda_buyout / lambda_total)

    u = np.random.uniform(size=len(active_idx))
    is_refi = u < p_refi
    is_buyout = (~is_refi) & (u < (p_refi + p_buyout))

    for j, i in enumerate(active_idx):
        event = "none"
        if is_refi[j]:
            event = "refinance"
        elif is_buyout[j]:
            event = "buyout"

        rows.append(
            {
                "loan_id": int(i),
                "month": month,
                "fico": float(fico[i]),
                "ltv": float(ltv[i]),
                "rate_incentive": float(incentive[j]),
                "event": event,
                "event_refi": int(is_refi[j]),
                "event_buyout": int(is_buyout[j]),
                "smm_true": float(p_refi[j] + p_buyout[j]),
            }
        )

    alive[active_idx[is_refi | is_buyout]] = False

panel = pd.DataFrame(rows)
panel.head(), panel.shape


In [ ]:
print(panel['event'].value_counts())
print('Average true SMM:', panel['smm_true'].mean().round(4))


In [ ]:
refi_model = smf.glm(
    formula="event_refi ~ bs(rate_incentive, df=6, degree=3) + bs(fico, df=6, degree=3)",
    data=panel,
    family=sm.families.Binomial(),
).fit()

buyout_model = smf.glm(
    formula="event_buyout ~ bs(ltv, df=6, degree=3) + bs(fico, df=6, degree=3)",
    data=panel,
    family=sm.families.Binomial(),
).fit()

panel['p_refi_hat'] = refi_model.predict(panel)
panel['p_buyout_hat'] = buyout_model.predict(panel)
panel['smm_hat'] = 1 - (1 - panel['p_refi_hat']) * (1 - panel['p_buyout_hat'])

panel[['event_refi', 'event_buyout', 'p_refi_hat', 'p_buyout_hat', 'smm_hat']].head()


In [ ]:
print('Refi model pseudo-R2 (McFadden):', round(refi_model.pseudo_rsquared(kind='mcf'), 4))
print('Buyout model pseudo-R2 (McFadden):', round(buyout_model.pseudo_rsquared(kind='mcf'), 4))
print('Observed SMM:', round((panel['event_refi'] + panel['event_buyout']).mean(), 4))
print('Predicted SMM:', round(panel['smm_hat'].mean(), 4))


In [ ]:
curve_df = (
    panel.groupby(pd.cut(panel['rate_incentive'], bins=25), observed=False)
    .agg(rate_incentive=('rate_incentive', 'mean'), refi_rate=('event_refi', 'mean'))
    .dropna()
)

fig = px.scatter(
    curve_df,
    x='rate_incentive',
    y='refi_rate',
    trendline='ols',
    title='Empirical S-curve: Rate Incentive vs Refinance Probability'
)
fig.show()


In [ ]:
ltv_df = (
    panel.groupby(pd.cut(panel['ltv'], bins=20), observed=False)
    .agg(ltv=('ltv', 'mean'), buyout_rate=('event_buyout', 'mean'))
    .dropna()
)

plt.figure(figsize=(7, 4))
plt.plot(ltv_df['ltv'], ltv_df['buyout_rate'], marker='o')
plt.title('Buyout Probability vs LTV')
plt.xlabel('LTV')
plt.ylabel('Buyout probability')
plt.grid(alpha=0.3)
plt.show()


### Notes
- FICO is clipped to [300, 850], centered near 714.
- LTV is clipped to [0, 1], centered near 0.75.
- Competing hazards are combined into a monthly SMM estimate.
- The two fitted submodels can be replaced by GAMs in R (e.g., `mgcv::gam`) via `rpy2` if desired.
